In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [4]:
import logging
import pandas as pd
import numpy as np
import awswrangler as wr
from matplotlib import pyplot as plt
import seaborn as sns
from general_functions.datetime_helper import transform_date_to_timestamp_milliseconds
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [5]:
customer = "Clockin"
goal_name = "tagmanagerregistercompleted"
from_date = "2026-01-05"
to_date = "2026-02-24"
intervall = 10
date_range = pd.date_range(start=from_date, end=to_date, freq=f"{intervall}D").strftime("%Y-%m-%d").tolist()
url = return_api_url()
print(f"url = {url}")
account_id = return_workspace_ids()
account_id = [acc["id"] for acc in account_id if acc["name"] == customer]
account_id = account_id[0]
date_range

In [6]:
conversions=pd.DataFrame()
list_dates = pd.date_range(from_date, to_date).strftime("%Y-%m-%d")
for idate, date in enumerate(list_dates):
    if idate == len(list_dates)-1:
        end_date = list_dates[-1]
    else:
        end_date = list_dates[idate+1]
    print(f"Query conversions from {date} to {end_date}")
    content={
        "created": {
                        "$gte": transform_date_to_timestamp_milliseconds(date),
                        "$lte": transform_date_to_timestamp_milliseconds(end_date),
                },
                "name":goal_name
                    }
    temp=send_to_innkeepr_api_paginated(
        f"{url}/conversions/query",
        account_id,
        content,
        logging
    )
    temp = pd.json_normalize(temp)
    print(temp)
    if temp.empty:
        print(".... no conversions for that date")
    conversions = pd.concat([conversions, temp])
conversions

In [11]:
print(f"Loaded conversions from {conversions['created'].min()} to {conversions['created'].max()}")
vc_conversions = conversions.groupby("name")["sessionId"].nunique()
vc_conversions

In [7]:
# query unique sessions
session_ids = conversions["sessionId"].dropna().unique().tolist()
print(f"Unique sessions: {len(session_ids)}")
sessions=pd.DataFrame()
for i in range(0, len(session_ids), 1000):
    end = i+1000
    if end >=  len(session_ids):
        end = len(session_ids)
    print(f"Query sessions from {i} to {end}")
    content={
        "sessionId": session_ids[i:end]}
    temp=send_to_innkeepr_api_paginated(
        f"{url}/sessions/query",
        account_id,
        content,
        logging
    )
    temp = pd.json_normalize(temp)
    if temp.empty:
        print(".... no sessions for that list")
    sessions = pd.concat([sessions, temp])
sessions

In [12]:
sessions["referrer"].str.contains("hsa_cam").sum()